# NB02: Data Transformation

This notebook turns the raw data file saved in the notebook 1 into a tidy table, with the necessary variables extracted and prepared for the analysis in notebook 3.



## Setup

In [2]:
import pandas as pd
import json
import os

### Step 1

Loading the raw files and keeping only the usable matches. 

The empty rows are dropped given that they do not provide any statistics, as specified in `NB01-Data-Collection.ipynb`,

In [3]:
file_path = '../data/raw/dump.json'

df = pd.read_json(file_path)

# empty list is false, so can be filtered out with bool
df_filtered1 = df[df["statistics"].apply(bool)].reset_index(drop=True)

df_filtered1

,first_player_key,second_player_key,event_winner,tournament_name,statistics
0,396,1742,First Player,Australian Open,"[{'player_key': 396, 'stat_period': 'match', '..."
1,424,9209,Second Player,Australian Open,"[{'player_key': 424, 'stat_period': 'match', '..."
2,39913,1095,First Player,Australian Open,"[{'player_key': 39913, 'stat_period': 'match',..."
3,3319,1082,Second Player,Australian Open,"[{'player_key': 3319, 'stat_period': 'match', ..."
4,7912,3696,Second Player,Australian Open,"[{'player_key': 7912, 'stat_period': 'match', ..."
...,...,...,...,...,...
707,372,8174,Second Player,Wimbledon,"[{'player_key': 372, 'stat_period': 'match', '..."
708,2832,1980,Second Player,Wimbledon,"[{'player_key': 2832, 'stat_period': 'match', ..."
709,2072,1905,First Player,Wimbledon,"[{'player_key': 2072, 'stat_period': 'match', ..."
710,8174,1980,Second Player,Wimbledon,"[{'player_key': 8174, 'stat_period': 'match', ..."


Each row in the dataset now represents one match. 

We have 712 rows as expected:

728 rows - 11 early withdrawals (that were replaced with alternate players) = 717 matches supposed to be played.

717 matches - 5 withdrawals late in the tournament = 712 rows.

### Step 2
Create the winner/loser columns. This is necessary as we are interested in the difference in number winners/unforced errors between the winner and loser, and to identify which is more important in determining who wins and loses. 

In [4]:
df_filtered1["event_winner"].unique() # initial sanity check to see what values are in the column

<ArrowStringArray>
['First Player', 'Second Player']
Length: 2, dtype: str

In [ ]:
df_filtered1a = df_filtered1[df_filtered1["event_winner"] == "First Player"].copy() # 1a: matches where the first player won
df_filtered1b = df_filtered1[df_filtered1["event_winner"] == "Second Player"].copy() # 1b: matches where the second player won

# rename the player keys to who won and lost in case 1a
df_filtered1a["won"] = df_filtered1a["first_player_key"] 
df_filtered1a["lost"] = df_filtered1a["second_player_key"]

# reversed for 1b
df_filtered1b["won"] = df_filtered1b["second_player_key"]
df_filtered1b["lost"] = df_filtered1b["first_player_key"]

# recombine now that the won/lost columns are consistent
df_filtered2 = pd.concat([df_filtered1a, df_filtered1b], ignore_index=True)


### Step 3
Filter out to only the statistics required for the analysis. Our statistics is currently a list of dictionaries like so:

In [6]:
df_filtered2["statistics"][0]

[{'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': 'Aces',
  'stat_value': '0',
  'stat_won': None,
  'stat_total': None},
 {'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': 'Double Faults',
  'stat_value': '2',
  'stat_won': None,
  'stat_total': None},
 {'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': '1st serve percentage',
  'stat_value': '64%',
  'stat_won': None,
  'stat_total': None},
 {'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': '1st serve points won',
  'stat_value': '68%',
  'stat_won': 30,
  'stat_total': 44},
 {'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': '2nd serve points won',
  'stat_value': '64%',
  'stat_won': 16,
  'stat_total': 25},
 {'player_key': 396,
  'stat_period': 'match',
  'stat_type': 'Service',
  'stat_name': 'Break Points Saved',
  'stat_value': '67%',
  'stat_won'

In [ ]:
# helper function to extract the statistics of interest

def extract_stats(nest):
    # keep only the 4 stats we are interested in
    kept = []
    keep = ["Winners", "Unforced errors", "Aces", "Double Faults"]
    
    for item in nest:
        # only keep stats for the whole match, not the individual sets
        if item["stat_period"] == "match":
            if item["stat_name"] in keep:
                kept.append(item)

    # build a dictionary with form {player_key: {stat_name: stat_value}}
    stat_dict = {}
    for entry in kept:
        # create new dict if it doesn't exist yet
        if entry["player_key"] not in stat_dict.keys():
            stat_dict[entry["player_key"]] = {}
    
        stat_dict[entry["player_key"]][entry["stat_name"]] = entry["stat_value"]

    return stat_dict

In [8]:
df_filtered3 = df_filtered2.copy()

# apply our helper function to the statistics column
df_filtered3["statistics"] = df_filtered2["statistics"].apply(extract_stats)

So now the entries in the statistics column are formatted like this:

In [9]:
df_filtered3["statistics"][0]

{396: {'Aces': '0',
  'Double Faults': '2',
  'Winners': '14',
  'Unforced errors': '19'},
 1742: {'Aces': '6',
  'Double Faults': '4',
  'Winners': '22',
  'Unforced errors': '35'}}

### Step 4
Create new columns using the values in those entries.

In [10]:
def process_stats(row):
    winner = row["won"]
    loser = row["lost"]
    stats = row["statistics"]
    
    to_return = {"winner_winners": int(stats[winner]["Winners"]),
                "winner_ue": int(stats[winner]["Unforced errors"]),
                "winner_aces": int(stats[winner]["Aces"]),
                "winner_df": int(stats[winner]["Double Faults"]),
                "loser_winners": int(stats[loser]["Winners"]),
                "loser_ue": int(stats[loser]["Unforced errors"]),  
                "loser_aces": int(stats[loser]["Aces"]), 
                "loser_df":int(stats[loser]["Double Faults"]), 
                }

    # return as a pandas series so they can be easily added as columns
    return pd.Series(to_return)

In [11]:
df_filtered4 = df_filtered3.copy()

# axis = 1 so the function is applied row-wise rather than to a single column
df_filtered4[["winner_winners", "winner_ue", "winner_aces", "winner_df", "loser_winners", "loser_ue", "loser_aces", "loser_df"]] = df_filtered3.apply(process_stats, axis=1)

### Step 5
Clean table by removing columns that are not required anymore. The data is now ready for analysis.

In [14]:
df_filtered5 = df_filtered4.drop(columns = ["event_winner", "first_player_key", "second_player_key", "statistics", "won", "lost"])

df_filtered5

,tournament_name,winner_winners,winner_ue,winner_aces,winner_df,loser_winners,loser_ue,loser_aces,loser_df
0,Australian Open,14,19,0,2,22,35,6,4
1,Australian Open,16,7,4,0,14,27,3,3
2,Australian Open,37,28,6,2,21,31,1,4
3,Australian Open,20,20,2,1,9,29,1,3
4,Australian Open,8,13,1,1,11,29,2,2
...,...,...,...,...,...,...,...,...,...
707,Wimbledon,36,33,7,0,72,59,19,7
708,Wimbledon,43,47,14,6,74,61,29,5
709,Wimbledon,27,15,8,1,21,41,6,2
710,Wimbledon,29,15,14,2,31,28,17,2


### Step 6
Output to csv, so that the data can be accessed in `NB03-tynelee-Data-Analysis`.

In [31]:
df_filtered5.to_csv("../data/processed/cleaned.csv", index=False)